# Oceananigans validation of the flat-bottom PG Green's-function solution

This notebook compares a weak, horizontally uniform, rotating Boussinesq column computed with Oceananigans against the analytical Green's-function solution. The flat bottom and periodic horizontal directions isolate the shared vertical physics: no-slip bottom velocity, surface stress, prescribed surface buoyancy, and interior relaxation toward `B(z)`.

Oceananigans solves the full nonhydrostatic Boussinesq equations, not the planetary-geostrophic reduction. The comparison is therefore an asymptotic weak-flow test. It deliberately uses no horizontal buoyancy gradient or surface-pressure gradient; a closed-basin/topographic comparison is a separate validation problem.

Oceananigans emits an upstream support warning on Julia 1.13 because it is tested through Julia 1.12. This CPU notebook is smoke-tested here on Julia 1.13; retain the warning with benchmark results.

In [ ]:
using Oceananigans
using LinearAlgebra
using Printf
using Plots
using Statistics

include(normpath(joinpath(pwd(), "..", "TheoryNotes", "VelocityFieldPrototype.jl")))
using .VelocityFieldPrototype


## Matched weak-flow case

The coefficients below use the same nondimensional convention as `PG_Solution_v0.1.ipynb`: Oceananigans viscosity and diffusivity are `Ek * nu` and `Ek * kappa`, respectively. Oceananigans surface momentum flux is kinematic and uses the opposite sign to the notebook's positive `∂z u = τx` convention, so it is `-nu_ocean * tau_analytical`.

In [ ]:
const Lx = 1.0
const Ly = 1.0
const H = 1.0
const f0 = 1.0
const Ek = 0.02
const Γ = 0.2
const nu_kernel = 1.0
const kappa_kernel = 1.0
const nu_ocean = Ek * nu_kernel
const kappa_ocean = Ek * kappa_kernel

# Keep the Rossby number small: the analytical solution has no inertia.
const tau_analytical = 1e-4 + 0.0im
const b_surface = 2e-3
const B_gradient = 1e-2

Hfun(x, y) = H
ffun(x, y) = f0
zero2(x, y) = 0.0
Bfun(x, y, z) = B_gradient * z
bsfun(x, y) = b_surface

parameters = VelocityParameters(Ek, Γ, nu_kernel, kappa_kernel, f0, 0.0)
kernels = load_symbolic_kernels()

b_reference(z) = buoyancy_from_forcing(parameters, Hfun, Bfun, bsfun, 0.0, 0.0, z)
u_reference(z) = velocity_b23_from_gradps(
    kernels, parameters, Hfun, ffun, zero2, zero2, zero2,
    (x, y) -> tau_analytical, Bfun, bsfun, zero2, zero2, 0.0, 0.0, z,
)

# Oceananigans calls continuous boundary functions with x, y, z.
surface_buoyancy(x, y, z) = b_surface
target_buoyancy(x, y, z, t) = B_gradient * z

@show nu_ocean kappa_ocean Γ tau_analytical b_surface


In [ ]:
function stable_timestep(Nz)
    dz = H / Nz
    return min(0.01, 0.2 * dz^2 / max(nu_ocean, kappa_ocean))
end

function make_simulation(Nz; Nx=8, Ny=8, Δt=stable_timestep(Nz), stop_time=20.0)
    grid = RectilinearGrid(CPU();
        topology = (Periodic, Periodic, Bounded),
        size = (Nx, Ny, Nz),
        x = (0, Lx), y = (0, Ly), z = (-H, 0),
    )

    u_bcs = FieldBoundaryConditions(
        top = FluxBoundaryCondition(-nu_ocean * real(tau_analytical)),
        bottom = ValueBoundaryCondition(0.0),
    )
    v_bcs = FieldBoundaryConditions(
        top = FluxBoundaryCondition(-nu_ocean * imag(tau_analytical)),
        bottom = ValueBoundaryCondition(0.0),
    )
    b_bcs = FieldBoundaryConditions(
        top = ValueBoundaryCondition(surface_buoyancy),
        bottom = GradientBoundaryCondition(0.0),
    )

    model = NonhydrostaticModel(grid;
        advection = nothing,
        buoyancy = BuoyancyTracer(),
        coriolis = FPlane(f=f0),
        tracers = :b,
        closure = ScalarDiffusivity(ν=nu_ocean, κ=kappa_ocean),
        boundary_conditions = (u=u_bcs, v=v_bcs, b=b_bcs),
        forcing = (b=Relaxation(rate=Γ, target=target_buoyancy),),
    )

    # Starting from the analytical buoyancy profile isolates momentum spin-up.
    set!(model, u=0, v=0, w=0, b=(x, y, z) -> b_reference(z))
    return Simulation(model; Δt, stop_time)
end

function relative_l2(numerical, reference)
    return norm(numerical .- reference) / max(norm(reference), eps(eltype(numerical)))
end

function compare_with_green_function(simulation)
    model = simulation.model
    u = Array(interior(model.velocities.u))
    v = Array(interior(model.velocities.v))
    b = Array(interior(model.tracers.b))
    z = collect(znodes(model.tracers.b))

    u_exact = reshape(real.(u_reference.(z)), 1, 1, :)
    v_exact = reshape(imag.(u_reference.(z)), 1, 1, :)
    b_exact = reshape(b_reference.(z), 1, 1, :)

    metrics = (
        u_relative_l2 = relative_l2(u, u_exact),
        v_relative_l2 = relative_l2(v, v_exact),
        b_relative_l2 = relative_l2(b, b_exact),
        max_speed = maximum(sqrt.(u.^2 .+ v.^2)),
    )
    @show metrics
    return metrics
end


## Accuracy and performance tests

The warm-up run compiles Oceananigans and is intentionally excluded from the timing. The Green's-function profile path is warmed separately. Its resources represent one vertical column, which is the appropriate analytical work for this horizontally uniform case; Oceananigans resources cover the full three-dimensional numerical domain. Increase `spinup_time` if velocity errors have not plateaued. Compare the error trend over `Nz` rather than relying on a single tolerance, because Oceananigans uses a staggered finite-volume discretization while the reference is continuous.

In [ ]:
const spinup_time = 100.0
const test_resolutions = (8, 16, 32, 64, 128)

# JIT warm-up is deliberately excluded from all reported resource timings.
warmup = make_simulation(first(test_resolutions); stop_time=0.1)
run!(warmup)

function timed_resources(f)
    GC.gc()
    measurement = @timed f()
    resources = (
        seconds = measurement.time,
        allocated_mib = measurement.bytes / 2.0^20,
        gc_seconds = measurement.gctime,
    )
    return measurement.value, resources
end

function green_function_profiles(z)
    velocity = u_reference.(z)
    return (u = real.(velocity), v = imag.(velocity), b = b_reference.(z))
end

green_function_profiles(collect(znodes(warmup.model.tracers.b)))

function mean_vertical_profile(field)
    values = Array(interior(field))
    return vec(dropdims(mean(values; dims=(1, 2)); dims=(1, 2)))
end

function benchmark_case(Nz; Nx=8, Ny=8, stop_time=spinup_time)
    simulation, ocean_resources = timed_resources() do
        simulation = make_simulation(Nz; Nx, Ny, stop_time)
        run!(simulation)
        simulation
    end

    z = collect(znodes(simulation.model.tracers.b))
    _, green_resources = timed_resources(() -> green_function_profiles(z))
    metrics = compare_with_green_function(simulation)
    return (
        Nz,
        cells = Nx * Ny * Nz,
        metrics,
        ocean_resources,
        green_resources,
    ), simulation
end

benchmark_cases = [benchmark_case(Nz) for Nz in test_resolutions]
benchmark_results = first.(benchmark_cases)
plot_simulation = last(benchmark_cases[end])

diagnostics_table = [
    (
        Nz = result.Nz,
        cells = result.cells,
        u_relative_l2 = result.metrics.u_relative_l2,
        v_relative_l2 = result.metrics.v_relative_l2,
        b_relative_l2 = result.metrics.b_relative_l2,
        max_speed = result.metrics.max_speed,
        ocean_seconds = result.ocean_resources.seconds,
        ocean_allocated_mib = result.ocean_resources.allocated_mib,
        ocean_gc_seconds = result.ocean_resources.gc_seconds,
        green_seconds = result.green_resources.seconds,
        green_allocated_mib = result.green_resources.allocated_mib,
        green_gc_seconds = result.green_resources.gc_seconds,
    )
    for result in benchmark_results
]
display(diagnostics_table)

case_parameters = (
    domain = (Lx=Lx, Ly=Ly, H=H, topology=(Periodic, Periodic, Bounded)),
    dynamics = (f0=f0, Ek=Ek, Γ=Γ, nu_kernel=nu_kernel, kappa_kernel=kappa_kernel,
                nu_ocean=nu_ocean, kappa_ocean=kappa_ocean),
    forcing = (tau_analytical=tau_analytical, surface_buoyancy=b_surface,
               B_gradient=B_gradient),
    boundary_conditions = (bottom_velocity=:no_slip, bottom_buoyancy=:zero_gradient,
                           top_stress=:kinematic_flux, top_buoyancy=:Dirichlet),
    numerical = (Nx=8, Ny=8, resolutions=test_resolutions, spinup_time=spinup_time,
                 timestep_rule="min(0.01, 0.2 * dz^2 / max(nu_ocean, kappa_ocean))"),
)
display(case_parameters)
@info "Benchmark environment" julia_version=VERSION oceananigans_version=Base.pkgversion(Oceananigans) threads=Threads.nthreads() case_parameters

z = collect(znodes(plot_simulation.model.tracers.b))
reference = green_function_profiles(z)
u_numerical = mean_vertical_profile(plot_simulation.model.velocities.u)
v_numerical = mean_vertical_profile(plot_simulation.model.velocities.v)
b_numerical = mean_vertical_profile(plot_simulation.model.tracers.b)

velocity_profile_plot = plot(
    u_numerical, z; label="Oceananigans u", xlabel="velocity", ylabel="z",
    title="Velocity profiles at Nz=$(last(test_resolutions))", linewidth=2,
)
plot!(velocity_profile_plot, reference.u, z; label="Green's u", linestyle=:dash, linewidth=2)
plot!(velocity_profile_plot, v_numerical, z; label="Oceananigans v", linewidth=2)
plot!(velocity_profile_plot, reference.v, z; label="Green's v", linestyle=:dash, linewidth=2)

buoyancy_profile_plot = plot(
    b_numerical, z; label="Oceananigans b", xlabel="buoyancy", ylabel="z",
    title="Buoyancy profile at Nz=$(last(test_resolutions))", linewidth=2,
)
plot!(buoyancy_profile_plot, reference.b, z; label="Green's b", linestyle=:dash, linewidth=2)

Nz = getproperty.(benchmark_results, :Nz)
error_plot = plot(Nz, getproperty.(getproperty.(benchmark_results, :metrics), :u_relative_l2);
    label="u", marker=:circle, yscale=:log10, xlabel="Nz", ylabel="relative L2 error",
    title="Green's-function error convergence",
)
plot!(error_plot, Nz, getproperty.(getproperty.(benchmark_results, :metrics), :v_relative_l2); label="v", marker=:circle)
plot!(error_plot, Nz, getproperty.(getproperty.(benchmark_results, :metrics), :b_relative_l2); label="b", marker=:circle)

runtime_plot = plot(Nz, getproperty.(getproperty.(benchmark_results, :ocean_resources), :seconds);
    label="Oceananigans", marker=:circle, yscale=:log10, xlabel="Nz", ylabel="seconds",
    title="Post-warm-up elapsed time",
)
plot!(runtime_plot, Nz, getproperty.(getproperty.(benchmark_results, :green_resources), :seconds); label="Green's profiles", marker=:circle)

memory_plot = plot(Nz, getproperty.(getproperty.(benchmark_results, :ocean_resources), :allocated_mib);
    label="Oceananigans", marker=:circle, yscale=:log10, xlabel="Nz", ylabel="allocated MiB",
    title="Allocated memory per case",
)
plot!(memory_plot, Nz, getproperty.(getproperty.(benchmark_results, :green_resources), :allocated_mib); label="Green's profiles", marker=:circle)

diagnostic_figure = plot(
    velocity_profile_plot, buoyancy_profile_plot, error_plot, runtime_plot, memory_plot;
    layout=(3, 2), size=(1200, 1200),
)
display(diagnostic_figure)
savefig(diagnostic_figure, "Oceananigans_PG_Validation_diagnostics.png")
